In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

'''
Action space:

0: do nothing
1: fire left orientation engine
2: fire main engine
3: fire right orientation engine

Observation Space:

The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

Rewards:

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward: is increased/decreased the closer/further the lander is to the landing pad.

- is increased/decreased the slower/faster the lander is moving.
- is decreased the more the lander is tilted (angle not horizontal).
- is increased by 10 points for each leg that is in contact with the ground.
- is decreased by 0.03 points each frame a side engine is firing.
- is decreased by 0.3 points each frame the main engine is firing.
- The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.
- An episode is considered a solution if it scores at least 200 points.

Starting State:
The lander starts at the top center of the viewport with a random initial force applied to its center of mass.

Episode Termination:
The episode finishes if:
- the lander crashes (the lander body gets in contact with the moon);
- the lander gets outside of the viewport (x coordinate is greater than 1);
- the lander is not awake. From the Box2D docs, a body which is not awake is a body which doesn't move and doesn't collide with any other body:
'''

env = make_vec_env('LunarLander-v3', n_envs=16)

# env = gym.make('LunarLander-v3')

In [ ]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 256,
    n_epochs = 10,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.005,
    verbose=1, device='cpu')

In [ ]:
model.learn(total_timesteps=500_000)
model_name = "ppo-LunarLander-v3"
model.save(model_name)

In [ ]:
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")